# Jainish — BillSum Week 2 GPU checks

## Purpose
Check whether full fine-tuning, LoRA, DoRA, and 4-bit QLoRA
can perform 20 successful training updates on one Tesla T4 GPU,
then save and reload their checkpoints correctly.

## Results
All four methods passed the latest checks:
- 20 successful optimizer steps per method.
- Save/reload output difference: 0.0 for each method.
- All 14 BillSum tests passed.

These short runs check training compatibility.
They do not establish summary quality or select a learning rate.

## Session history
The first session ended before its report files were downloaded.
Its displayed outputs were retained in the notebook.
The checks were rerun to regenerate the reports, which were
downloaded in week2_evidence_all_four.zip.

The earlier QLoRA attempt exposed a frozen-bias dtype conversion.
The fix prepares those biases before integrity checks and verifies
that the conversion preserves their numerical values.

This notebook currently includes earlier diagnostic outputs and
the later reruns. Run cells individually; it is not yet arranged
for an uninterrupted Run All execution.

In [1]:
import os

# Use only the first GPU.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Python: 3.13.16
PyTorch: 2.11.0+cu130
GPU available: True
Visible GPUs: 1
GPU name: Tesla T4


In [2]:
from pathlib import Path
import subprocess

project = Path("/kaggle/working/lora-on-a-budget")

if not project.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/devjp99/lora-on-a-budget.git",
            str(project)
        ],
        check=True
    )
else:
    print("Project folder already exists; keeping it.")

print("\nCode version in this session:")
subprocess.run(
    ["git", "log", "-1", "--format=%h %s"],
    cwd=str(project),
    check=True
)

Cloning into '/kaggle/working/lora-on-a-budget'...



Code version in this session:
483f0fe Merge pull request #5 from devjp99/jainish/week2-billsum


CompletedProcess(args=['git', 'log', '-1', '--format=%h %s'], returncode=0)

In [3]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "torch",
    "transformers",
    "peft",
    "accelerate",
    "bitsandbytes",
    "torchao",
    "huggingface-hub",
    "pandas",
    "pyarrow",
    "PyYAML",
    "pytest"
]

for package in packages:
    try:
        print(package, ":", version(package))
    except PackageNotFoundError:
        print(package, ": not installed")

torch : 2.11.0+cu130
transformers : 5.20.2
peft : 0.21.2
accelerate : 1.15.0
bitsandbytes : not installed
torchao : 0.10.0
huggingface-hub : 1.33.0
pandas : 2.3.3
pyarrow : 23.0.1
PyYAML : 6.0.3
pytest : 8.4.2


In [4]:
setup_logs = Path("/kaggle/working/week2_evidence/setup")
setup_logs.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "torchao_removal.txt").write_text(output)
result.check_returncode()

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0



In [5]:
result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--editable", str(project),
        "--no-deps"
    ],
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "project_install.txt").write_text(output)
result.check_returncode()

Obtaining file:///kaggle/working/lora-on-a-budget
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for lora_budget (pyproject.toml): started
  Building editable for lora_budget (pyproject.toml): finished with status 'done'
  Created wheel for lora_budget: filename=lora_budget-0.1.0-0.editable-py3-none-any.whl size=1306 sha256=2b76e1bb448ecf2574668a1ca58e61428757eac2f90b4ac0c744592e458b783f
  Stored in directory: /tmp/pip-ephem-wheel-cache-ysrtl4mo/wheels/fc/7f/c1/d386dabe01896a581402da5602269710baff4d17b8d9b94a97
Successfully b

In [6]:
result = subprocess.run(
    [
        sys.executable,
        "scripts/make_billsum_manifest.py",
        "--verify-existing"
    ],
    cwd=str(project),
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "manifest_verification.txt").write_text(output)
result.check_returncode()

Locked raw files and all manifests verified



In [7]:
config_path = project / "configs/billsum/full_ft.yaml"

print(config_path.read_text())

# Candidate settings; resolved model SHA and manifest hash saved by each pilot.
experiment: E2
task: billsum
owner: Jainish Patel
method: full_ft
model:
  name: Qwen/Qwen2.5-0.5B
  revision: 060db6499f32faf8b98477b0a26969ef7d8b9987
data:
  raw_dir: data/raw/billsum
  manifest_dir: manifests/billsum
  max_prompt: 1024
  max_target: 256
  subset_seed: 2026
train:
  epochs: 3
  micro_batch: 1
  grad_accum: 16
  learning_rates:
  - 1.0e-05
  - 3.0e-05
  weight_decay: 0.01
  warmup_ratio: 0.06
  scheduler: linear
  max_grad_norm: 1.0
  precision: fp16
  seeds:
  - 42
  - 43
  - 44
  gradient_checkpointing: true
  use_cache: false
lora:
  rank: 8
  alpha: 16
  dropout: 0.05
  target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj
eval:
  select_by: tune_rougeL
  do_sample: false
  num_beams: 1
  max_new_tokens: 256



In [16]:
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log_path = setup_logs.parent / f"full_ft_{stamp}.txt"

command = [
    sys.executable, "-u",
    "scripts/pilot_billsum.py",
    "--config", "configs/billsum/full_ft.yaml",
    "--steps", "20",
    "--seed", "42"
]

with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=str(project),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    exit_code = process.wait()

print("\nExit code:", exit_code)
print("Console log saved:", log_path)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 557.98it/s]
1/20: loss=1.54399
2/20: loss=1.56039
3/20: loss=1.33352
4/20: loss=1.60941
5/20: loss=1.34548
6/20: loss=1.56346
7/20: loss=1.54067
8/20: loss=1.56914
9/20: loss=1.15503
10/20: loss=1.35936
11/20: loss=1.73648
12/20: loss=1.50375
13/20: loss=1.65044
14/20: loss=1.46518
15/20: loss=1.15554
16/20: loss=1.55389
17/20: loss=1.37090
18/20: loss=1.29218
19/20: loss=1.50242
20/20: loss=1.44373

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.39s/it]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2641.38it/s]
Recorded results/billsum/pilots/billsum-full_ft-4114c72ac3a7.json

Exit code: 0
Console log saved: /kaggle/working/week2_evidence/full_ft_20261112_060640.txt


In [9]:
import json

report_path = (
    project
    / "results/billsum/pilots/billsum-full_ft-65386329b978.json"
)

full_ft_report = json.loads(report_path.read_text())

fields = [
    "passed",
    "status",
    "successful_optimizer_steps",
    "loading_memory",
    "forward_backward_memory",
    "first_optimizer_memory",
    "steady_memory",
    "mean_steady_step_seconds",
    "checkpoint_bytes",
    "reload_max_abs_diff",
    "reload_tolerance"
]

for field in fields:
    print(field, ":", full_ft_report.get(field, "not recorded"))

passed : True
status : completed
successful_optimizer_steps : 20
loading_memory : {'allocated_bytes': 1976873472, 'reserved_bytes': 2097152000}
forward_backward_memory : {'allocated_bytes': 7057676288, 'reserved_bytes': 8378122240}
first_optimizer_memory : {'allocated_bytes': 9912224256, 'reserved_bytes': 10202644480}
steady_memory : {'allocated_bytes': 11023108096, 'reserved_bytes': 12471762944}
mean_steady_step_seconds : 12.800458151800004
checkpoint_bytes : 1987589933
reload_max_abs_diff : 0.0
reload_tolerance : 0.001


In [10]:
print(
    (project / "configs/billsum/lora.yaml").read_text()
)

# Candidate settings; resolved model SHA and manifest hash saved by each pilot.
experiment: E2
task: billsum
owner: Jainish Patel
method: lora
model:
  name: Qwen/Qwen2.5-0.5B
  revision: 060db6499f32faf8b98477b0a26969ef7d8b9987
data:
  raw_dir: data/raw/billsum
  manifest_dir: manifests/billsum
  max_prompt: 1024
  max_target: 256
  subset_seed: 2026
train:
  epochs: 3
  micro_batch: 1
  grad_accum: 16
  learning_rates:
  - 0.0001
  - 0.0003
  weight_decay: 0.01
  warmup_ratio: 0.06
  scheduler: linear
  max_grad_norm: 1.0
  precision: fp16
  seeds:
  - 42
  - 43
  - 44
  gradient_checkpointing: true
  use_cache: false
lora:
  rank: 8
  alpha: 16
  dropout: 0.05
  target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj
eval:
  select_by: tune_rougeL
  do_sample: false
  num_beams: 1
  max_new_tokens: 256



In [11]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log_path = setup_logs.parent / f"lora_{stamp}.txt"

command = [
    sys.executable, "-u",
    "scripts/pilot_billsum.py",
    "--config", "configs/billsum/lora.yaml",
    "--steps", "20",
    "--seed", "42"
]

with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=str(project),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    exit_code = process.wait()

print("\nExit code:", exit_code)
print("Console log saved:", log_path)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 620.29it/s]
1/20: loss=1.54422
2/20: loss=1.56047
3/20: loss=1.39999
4/20: loss=1.69448
5/20: loss=1.44985
6/20: loss=1.63392
7/20: loss=1.65394
8/20: loss=1.64516
9/20: loss=1.70335
10/20: loss=1.25590
11/20: loss=1.44911
12/20: loss=1.81344
13/20: loss=1.58617
14/20: loss=1.70508
15/20: loss=1.56832
16/20: loss=1.28459
17/20: loss=1.64025
18/20: loss=1.46109
19/20: loss=1.37334
20/20: loss=1.58556

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 611.68it/s]
Recorded results/billsum/pilots/billsum-lora-7a3fd2652a56.json

Exit code: 0
Console log saved: /kaggle/working/week2_evidence/lora_20261111_215737.txt


In [12]:
lora_report_path = (
    project
    / "results/billsum/pilots/billsum-lora-7a3fd2652a56.json"
)

lora_report = json.loads(lora_report_path.read_text())

fields = [
    "passed",
    "status",
    "successful_optimizer_steps",
    "loading_memory",
    "forward_backward_memory",
    "first_optimizer_memory",
    "steady_memory",
    "mean_steady_step_seconds",
    "checkpoint_bytes",
    "reload_max_abs_diff",
    "reload_tolerance"
]

for field in fields:
    print(field, ":", lora_report.get(field, "not recorded"))

passed : True
status : completed
successful_optimizer_steps : 20
loading_memory : {'allocated_bytes': 1013939200, 'reserved_bytes': 1056964608}
forward_backward_memory : {'allocated_bytes': 3446782464, 'reserved_bytes': 4108320768}
first_optimizer_memory : {'allocated_bytes': 3446782464, 'reserved_bytes': 4150263808}
steady_memory : {'allocated_bytes': 3482987008, 'reserved_bytes': 4150263808}
mean_steady_step_seconds : 12.774836767000004
checkpoint_bytes : 29071512
reload_max_abs_diff : 0.0
reload_tolerance : 0.001


In [13]:
print(
    (project / "configs/billsum/dora.yaml").read_text()
)

# Candidate settings; resolved model SHA and manifest hash saved by each pilot.
experiment: E2
task: billsum
owner: Jainish Patel
method: dora
model:
  name: Qwen/Qwen2.5-0.5B
  revision: 060db6499f32faf8b98477b0a26969ef7d8b9987
data:
  raw_dir: data/raw/billsum
  manifest_dir: manifests/billsum
  max_prompt: 1024
  max_target: 256
  subset_seed: 2026
train:
  epochs: 3
  micro_batch: 1
  grad_accum: 16
  learning_rates:
  - 0.0001
  - 0.0003
  weight_decay: 0.01
  warmup_ratio: 0.06
  scheduler: linear
  max_grad_norm: 1.0
  precision: fp16
  seeds:
  - 42
  - 43
  - 44
  gradient_checkpointing: true
  use_cache: false
lora:
  rank: 8
  alpha: 16
  dropout: 0.05
  target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj
eval:
  select_by: tune_rougeL
  do_sample: false
  num_beams: 1
  max_new_tokens: 256



In [14]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log_path = setup_logs.parent / f"dora_{stamp}.txt"

command = [
    sys.executable, "-u",
    "scripts/pilot_billsum.py",
    "--config", "configs/billsum/dora.yaml",
    "--steps", "20",
    "--seed", "42"
]

with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=str(project),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    exit_code = process.wait()

print("\nExit code:", exit_code)
print("Console log saved:", log_path)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 677.30it/s]
1/20: loss=1.54409
2/20: loss=1.56071
3/20: loss=1.39341
4/20: loss=1.68953
5/20: loss=1.44567
6/20: loss=1.62952
7/20: loss=1.64961
8/20: loss=1.63928
9/20: loss=1.69609
10/20: loss=1.24867
11/20: loss=1.44149
12/20: loss=1.80795
13/20: loss=1.58082
14/20: loss=1.70051
15/20: loss=1.56269
16/20: loss=1.27769
17/20: loss=1.63526
18/20: loss=1.45576
19/20: loss=1.36828
20/20: loss=1.58002

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 679.54it/s]
Recorded results/billsum/pilots/billsum-dora-15923a89e593.json

Exit code: 0
Console log saved: /kaggle/working/week2_evidence/dora_20261111_220430.txt


In [15]:
dora_report_path = (
    project
    / "results/billsum/pilots/billsum-dora-15923a89e593.json"
)

dora_report = json.loads(dora_report_path.read_text())

fields = [
    "passed",
    "status",
    "successful_optimizer_steps",
    "loading_memory",
    "forward_backward_memory",
    "first_optimizer_memory",
    "steady_memory",
    "mean_steady_step_seconds",
    "checkpoint_bytes",
    "reload_max_abs_diff",
    "reload_tolerance"
]

for field in fields:
    print(field, ":", dora_report.get(field, "not recorded"))

passed : True
status : completed
successful_optimizer_steps : 20
loading_memory : {'allocated_bytes': 1015155712, 'reserved_bytes': 1059061760}
forward_backward_memory : {'allocated_bytes': 3449654784, 'reserved_bytes': 4162846720}
first_optimizer_memory : {'allocated_bytes': 3449654784, 'reserved_bytes': 4206886912}
steady_memory : {'allocated_bytes': 3487670784, 'reserved_bytes': 4206886912}
mean_steady_step_seconds : 22.76464564233338
checkpoint_bytes : 30310927
reload_max_abs_diff : 0.0
reload_tolerance : 0.001


In [16]:
print(
    (project / "configs/billsum/qlora.yaml").read_text()
)

# Candidate settings; resolved model SHA and manifest hash saved by each pilot.
experiment: E2
task: billsum
owner: Jainish Patel
method: qlora
model:
  name: Qwen/Qwen2.5-0.5B
  revision: 060db6499f32faf8b98477b0a26969ef7d8b9987
data:
  raw_dir: data/raw/billsum
  manifest_dir: manifests/billsum
  max_prompt: 1024
  max_target: 256
  subset_seed: 2026
train:
  epochs: 3
  micro_batch: 1
  grad_accum: 16
  learning_rates:
  - 0.0001
  - 0.0003
  weight_decay: 0.01
  warmup_ratio: 0.06
  scheduler: linear
  max_grad_norm: 1.0
  precision: fp16
  seeds:
  - 42
  - 43
  - 44
  gradient_checkpointing: true
  use_cache: false
lora:
  rank: 8
  alpha: 16
  dropout: 0.05
  target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj
eval:
  select_by: tune_rougeL
  do_sample: false
  num_beams: 1
  max_new_tokens: 256



In [17]:
result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "bitsandbytes==0.50.2",
        "--only-binary=:all:",
        "--no-deps"
    ],
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "bitsandbytes_install.txt").write_text(output)
result.check_returncode()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 72.8 MB/s  0:00:00



In [18]:
check_code = """
import torch
import bitsandbytes as bnb

print("bitsandbytes:", bnb.__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())

assert torch.cuda.is_available(), "GPU is not available"
assert torch.cuda.device_count() == 1, "Expected exactly one visible GPU"

print("GPU name:", torch.cuda.get_device_name(0))
"""

result = subprocess.run(
    [sys.executable, "-c", check_code],
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "bitsandbytes_import_check.txt").write_text(output)
result.check_returncode()

bitsandbytes: 0.50.2
PyTorch: 2.11.0+cu130
GPU available: True
Visible GPUs: 1
GPU name: Tesla T4



In [19]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log_path = setup_logs.parent / f"qlora_{stamp}.txt"

command = [
    sys.executable, "-u",
    "scripts/pilot_billsum.py",
    "--config", "configs/billsum/qlora.yaml",
    "--steps", "20",
    "--seed", "42"
]

with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=str(project),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    exit_code = process.wait()

print("\nExit code:", exit_code)
print("Console log saved:", log_path)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 351.12it/s]
1/20: loss=1.63331
2/20: loss=1.64276
3/20: loss=1.48325
4/20: loss=1.76126
5/20: loss=1.51677
6/20: loss=1.72074
7/20: loss=1.73374
8/20: loss=1.74607
9/20: loss=1.78593
10/20: loss=1.32840
11/20: loss=1.51294
12/20: loss=1.90046
13/20: loss=1.65936
14/20: loss=1.76572
15/20: loss=1.63009
16/20: loss=1.33302
17/20: loss=1.70196
18/20: loss=1.52081
19/20: loss=1.43203
20/20: loss=1.65959
Recorded results/billsum/pilots/billsum-qlora-fc7f36da54d7.json
Traceback (most recent call last):
  File "/kaggle/working/lora-on-a-budget/scripts/pilot_billsum.py", line 174, in <module>
    if __name__=='__main__': main()
                             ~~~~^^
  File "/kaggle/working/lora-on-a-budget/scripts/pilot_billsum.py", line 141, in main
    raise ValueError(f'Frozen parameter changed: {n}')
ValueError: Frozen parameter changed: base_model.model.model.layers.0.sel

In [20]:
diagnostic_code = """
import inspect
import bitsandbytes as bnb

print("bitsandbytes version:", bnb.__version__)
print(inspect.getsource(bnb.nn.Linear4bit.forward))
"""

result = subprocess.run(
    [sys.executable, "-c", diagnostic_code],
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "qlora_bias_diagnostic.txt").write_text(output)
result.check_returncode()

bitsandbytes version: 0.50.2
    def forward(self, x: torch.Tensor):
        fix_4bit_weight_quant_state_from_module(self)
        quant_state = self.weight.quant_state

        if (
            x.device.type == "cpu"
            and self.support_avx512bf16_for_cpu
            and not self.training
            and x.requires_grad == False
            and not getattr(quant_state, "packing_format_for_cpu", False)
        ):
            self.weight.data, quant_state = _convert_weight_packed_for_cpu(self.weight.data, quant_state)

        if not self.compute_type_is_set:
            self.set_compute_type(x)
            self.compute_type_is_set = True

        inp_dtype = x.dtype
        if self.compute_dtype is not None:
            x = x.to(self.compute_dtype)

        bias = self.bias
        if bias is not None:
            if bias.dtype != x.dtype:
                # TODO: do we need to cast bias like this?
                bias.data = bias.data.to(x.dtype)
            bias = bias.to(sel

In [21]:
diagnostic_code = """
import torch
from transformers import AutoTokenizer
from lora_budget.models.qwen import build_model
from lora_budget.data.billsum import load_split, encode_example

revision = "060db6499f32faf8b98477b0a26969ef7d8b9987"
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B", revision=revision
)

model = build_model("qlora", revision).eval()

name = "base_model.model.model.layers.0.self_attn.q_proj.base_layer.bias"
bias = dict(model.named_parameters())[name]
before = bias.detach().cpu().clone()

print("Trainable:", bias.requires_grad)
print("Before forward:", before.dtype)

example = encode_example(load_split("train")[0], tokenizer)
inputs = torch.tensor([example["input_ids"]], device="cuda")
mask = torch.ones_like(inputs)

with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    model(input_ids=inputs, attention_mask=mask)

after = bias.detach().cpu().clone()

print("After forward:", after.dtype)
print("Bytes unchanged:",
      before.numpy().tobytes() == after.numpy().tobytes())
print("Matches a dtype conversion:",
      torch.equal(before.to(after.dtype), after))
print("Maximum numerical difference:",
      (before.float() - after.float()).abs().max().item())
print("No backward pass or optimizer update was performed.")
"""

result = subprocess.run(
    [sys.executable, "-u", "-c", diagnostic_code],
    cwd=str(project),
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "qlora_bias_forward_check.txt").write_text(output)
result.check_returncode()

Trainable: False
Before forward: torch.float32
After forward: torch.float16
Bytes unchanged: False
Matches a dtype conversion: True
Maximum numerical difference: 0.0
No backward pass or optimizer update was performed.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 352.85it/s]



In [14]:
import difflib

model_file = project / "src/lora_budget/models/qwen.py"
original = model_file.read_text()

anchor = (
    "        model = prepare_model_for_kbit_training"
    "(model, use_gradient_checkpointing=True)\n"
)

addition = """        # Normalize frozen 4-bit biases before integrity fingerprints.
        from bitsandbytes.nn import Linear4bit
        for name, layer in model.named_modules():
            if isinstance(layer, Linear4bit) and layer.bias is not None:
                bias = layer.bias
                if bias.requires_grad:
                    raise ValueError(f"Expected frozen QLoRA bias: {name}")
                dtype = layer.compute_dtype
                if dtype is None:
                    raise ValueError(f"Missing QLoRA compute dtype: {name}")
                if bias.dtype != dtype:
                    converted = bias.detach().to(dtype)
                    if not torch.equal(
                        bias.detach().float(), converted.float()
                    ):
                        raise ValueError(
                            f"Bias conversion would change values: {name}"
                        )
                    with torch.no_grad():
                        bias.data = converted
"""

if "# Normalize frozen 4-bit biases" in original:
    print("The fix is already present; no changes made.")
else:
    assert original.count(anchor) == 1, "Expected setup line not found exactly once"

    updated = original.replace(anchor, anchor + addition, 1)

    (setup_logs / "qwen_before_bias_fix.py").write_text(original)

    patch = "".join(difflib.unified_diff(
        original.splitlines(keepends=True),
        updated.splitlines(keepends=True),
        fromfile="a/src/lora_budget/models/qwen.py",
        tofile="b/src/lora_budget/models/qwen.py"
    ))

    (setup_logs / "qlora_bias_fix.patch").write_text(patch)
    (setup_logs / "qwen_after_bias_fix.py").write_text(updated)
    model_file.write_text(updated)

    print(patch)
    print("Local QLoRA setup updated. Frozen-parameter checks remain enabled.")

The fix is already present; no changes made.


In [23]:
import shutil

original_log = setup_logs / "qlora_bias_forward_check.txt"
backup_log = setup_logs / "qlora_bias_forward_check_before_fix.txt"

if original_log.exists() and not backup_log.exists():
    shutil.copy2(original_log, backup_log)

print("Earlier diagnostic log preserved.")

Earlier diagnostic log preserved.


In [25]:
diagnostic_code = """
import torch
from transformers import AutoTokenizer
from lora_budget.models.qwen import build_model
from lora_budget.data.billsum import load_split, encode_example

revision = "060db6499f32faf8b98477b0a26969ef7d8b9987"
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B", revision=revision
)

model = build_model("qlora", revision).eval()

name = "base_model.model.model.layers.0.self_attn.q_proj.base_layer.bias"
bias = dict(model.named_parameters())[name]
before = bias.detach().cpu().clone()

print("Trainable:", bias.requires_grad)
print("Before forward:", before.dtype)

example = encode_example(load_split("train")[0], tokenizer)
inputs = torch.tensor([example["input_ids"]], device="cuda")
mask = torch.ones_like(inputs)

with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    model(input_ids=inputs, attention_mask=mask)

after = bias.detach().cpu().clone()

print("After forward:", after.dtype)
print("Bytes unchanged:",
      before.numpy().tobytes() == after.numpy().tobytes())
print("Matches a dtype conversion:",
      torch.equal(before.to(after.dtype), after))
print("Maximum numerical difference:",
      (before.float() - after.float()).abs().max().item())
print("No backward pass or optimizer update was performed.")
"""

result = subprocess.run(
    [sys.executable, "-u", "-c", diagnostic_code],
    cwd=str(project),
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "qlora_bias_forward_check_after_fix.txt").write_text(output)
result.check_returncode()

Trainable: False
Before forward: torch.float16
After forward: torch.float16
Bytes unchanged: True
Matches a dtype conversion: True
Maximum numerical difference: 0.0
No backward pass or optimizer update was performed.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 359.39it/s]



In [26]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log_path = setup_logs.parent / f"qlora_retry_{stamp}.txt"

command = [
    sys.executable, "-u",
    "scripts/pilot_billsum.py",
    "--config", "configs/billsum/qlora.yaml",
    "--steps", "20",
    "--seed", "42"
]

with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=str(project),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    exit_code = process.wait()

print("\nExit code:", exit_code)
print("Console log saved:", log_path)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 360.28it/s]
1/20: loss=1.63331
2/20: loss=1.64276
3/20: loss=1.48325
4/20: loss=1.76126
5/20: loss=1.51677
6/20: loss=1.72074
7/20: loss=1.73374
8/20: loss=1.74607
9/20: loss=1.78593
10/20: loss=1.32840
11/20: loss=1.51294
12/20: loss=1.90046
13/20: loss=1.65936
14/20: loss=1.76572
15/20: loss=1.63009
16/20: loss=1.33302
17/20: loss=1.70196
18/20: loss=1.52081
19/20: loss=1.43203
20/20: loss=1.65959

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 393.01it/s]
Recorded results/billsum/pilots/billsum-qlora-c19a2a71a212.json

Exit code: 0
Console log saved: /kaggle/working/week2_evidence/qlora_retry_20261111_233221.txt


In [5]:
from pathlib import Path
import json

project = Path("/kaggle/working/lora-on-a-budget")

report_path = (
    project
    / "results/billsum/pilots/billsum-qlora-c19a2a71a212.json"
)

print("Project folder exists:", project.is_dir())
print("QLoRA report exists:", report_path.is_file())

if report_path.is_file():
    qlora_report = json.loads(report_path.read_text())

    fields = [
        "passed",
        "status",
        "successful_optimizer_steps",
        "loading_memory",
        "forward_backward_memory",
        "first_optimizer_memory",
        "steady_memory",
        "mean_steady_step_seconds",
        "checkpoint_bytes",
        "reload_max_abs_diff",
        "reload_tolerance"
    ]

    for field in fields:
        print(field, ":", qlora_report.get(field, "not recorded"))
else:
    print("Report not found at the expected location. Stop here.")

Project folder exists: False
QLoRA report exists: False
Report not found at the expected location. Stop here.


In [6]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Python: 3.13.16
PyTorch: 2.11.0+cu130
GPU available: True
Visible GPUs: 1
GPU name: Tesla T4


In [7]:
from pathlib import Path
import subprocess

project = Path("/kaggle/working/lora-on-a-budget")

if not project.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/devjp99/lora-on-a-budget.git",
            str(project)
        ],
        check=True
    )
    subprocess.run(
        ["git", "checkout", "--detach", "483f0fe"],
        cwd=str(project),
        check=True
    )
else:
    print("Project folder already exists; leaving its files unchanged.")

print("\nCode version in this session:")
subprocess.run(
    ["git", "log", "-1", "--format=%h %s"],
    cwd=str(project),
    check=True
)

setup_logs = Path("/kaggle/working/week2_evidence/setup")
setup_logs.mkdir(parents=True, exist_ok=True)

print("\nEvidence folder ready:", setup_logs)

Cloning into '/kaggle/working/lora-on-a-budget'...



Code version in this session:
483f0fe Merge pull request #5 from devjp99/jainish/week2-billsum

Evidence folder ready: /kaggle/working/week2_evidence/setup


HEAD is now at 483f0fe Merge pull request #5 from devjp99/jainish/week2-billsum


In [8]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "torch", "transformers", "peft", "accelerate",
    "bitsandbytes", "torchao", "huggingface-hub",
    "pandas", "pyarrow", "PyYAML", "pytest"
]

lines = []

for package in packages:
    try:
        line = f"{package}: {version(package)}"
    except PackageNotFoundError:
        line = f"{package}: not installed"

    print(line)
    lines.append(line)

(setup_logs / "environment_before_setup.txt").write_text(
    "\n".join(lines) + "\n"
)

torch: 2.11.0+cu130
transformers: 5.20.2
peft: 0.21.2
accelerate: 1.15.0
bitsandbytes: not installed
torchao: 0.10.0
huggingface-hub: 1.33.0
pandas: 2.3.3
pyarrow: 23.0.1
PyYAML: 6.0.3
pytest: 8.4.2


199

In [9]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "torchao_removal.txt").write_text(output)
result.check_returncode()

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0



In [10]:
result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--editable", str(project),
        "--no-deps"
    ],
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "project_install.txt").write_text(output)
result.check_returncode()

Obtaining file:///kaggle/working/lora-on-a-budget
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for lora_budget (pyproject.toml): started
  Building editable for lora_budget (pyproject.toml): finished with status 'done'
  Created wheel for lora_budget: filename=lora_budget-0.1.0-0.editable-py3-none-any.whl size=1306 sha256=3c11a42669eeddbd7e46ce3a32ee8393ea285434f08dc048bc9938dece1854a2
  Stored in directory: /tmp/pip-ephem-wheel-cache-fh2hxevf/wheels/fc/7f/c1/d386dabe01896a581402da5602269710baff4d17b8d9b94a97
Successfully b

In [11]:
result = subprocess.run(
    [
        sys.executable,
        "scripts/make_billsum_manifest.py",
        "--verify-existing"
    ],
    cwd=str(project),
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "manifest_verification.txt").write_text(output)
result.check_returncode()

Locked raw files and all manifests verified



In [12]:
result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "bitsandbytes==0.50.2",
        "--only-binary=:all:",
        "--no-deps"
    ],
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "bitsandbytes_install.txt").write_text(output)
result.check_returncode()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 90.5 MB/s  0:00:00



In [15]:
result = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        "-q", "tests/test_billsum.py"
    ],
    cwd=str(project),
    capture_output=True,
    text=True
)

output = result.stdout + result.stderr
print(output)

(setup_logs / "billsum_tests.txt").write_text(output)
result.check_returncode()

..............                                                           [100%]
=============================== warnings summary ===============================
tests/test_billsum.py: 14 warnings
  /usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:365: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
14 passed, 14 warnings in 25.42s



In [18]:
import json
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

report_path = (
    project
    / "results/billsum/pilots/billsum-full_ft-4114c72ac3a7.json"
)
report = json.loads(report_path.read_text())

for field in [
    "passed",
    "status",
    "successful_optimizer_steps",
    "steady_memory",
    "mean_steady_step_seconds",
    "checkpoint_bytes",
    "reload_max_abs_diff",
    "reload_tolerance"
]:
    print(field, ":", report.get(field, "not recorded"))

zip_path = Path("/kaggle/working/week2_evidence_after_full_ft.zip")

with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for folder in [
        project / "results/billsum",
        Path("/kaggle/working/week2_evidence")
    ]:
        for file in sorted(folder.rglob("*")):
            if file.is_file():
                archive.write(
                    file,
                    file.relative_to("/kaggle/working")
                )

    archive.write(
        project / "src/lora_budget/models/qwen.py",
        "source_snapshot/qwen.py"
    )

print("\nBackup created:", zip_path)
print("Contains reports, logs, and current QLoRA model code.")
print("Model checkpoints and raw datasets are not included.")
display(FileLink(str(zip_path)))

passed : True
status : completed
successful_optimizer_steps : 20
steady_memory : {'allocated_bytes': 11023108096, 'reserved_bytes': 12471762944}
mean_steady_step_seconds : 14.388870364533355
checkpoint_bytes : 1987589933
reload_max_abs_diff : 0.0
reload_tolerance : 0.001

Backup created: /kaggle/working/week2_evidence_after_full_ft.zip
Contains reports, logs, and current QLoRA model code.
Model checkpoints and raw datasets are not included.


/kaggle/working/week2_evidence_after_full_ft.zip

## Latest rerun results

The latest full fine-tuning result appears above.
The latest LoRA, DoRA, and QLoRA runs follow below.

These are the four reports used for the Week 2 results:

| Method | Report filename |
|---|---|
| Full fine-tuning | billsum-full_ft-4114c72ac3a7.json |
| LoRA | billsum-lora-1564994f1226.json |
| DoRA | billsum-dora-5ccc88dbe43a.json |
| QLoRA | billsum-qlora-b6bcc21e3505.json |

Earlier outputs document the first session and debugging history.
They are not additional experimental repetitions.

All four latest reports and their console logs were downloaded
in week2_evidence_all_four.zip.

In [20]:
import subprocess
import sys
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log_path = setup_logs.parent / f"lora_{stamp}.txt"

command = [
    sys.executable, "-u",
    "scripts/pilot_billsum.py",
    "--config", "configs/billsum/lora.yaml",
    "--steps", "20",
    "--seed", "42"
]

with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=str(project),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    exit_code = process.wait()

print("\nExit code:", exit_code)
print("Console log saved:", log_path)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 694.14it/s]
1/20: loss=1.54422
2/20: loss=1.56047
3/20: loss=1.39999
4/20: loss=1.69448
5/20: loss=1.44985
6/20: loss=1.63392
7/20: loss=1.65394
8/20: loss=1.64516
9/20: loss=1.70335
10/20: loss=1.25590
11/20: loss=1.44911
12/20: loss=1.81344
13/20: loss=1.58617
14/20: loss=1.70508
15/20: loss=1.56832
16/20: loss=1.28459
17/20: loss=1.64025
18/20: loss=1.46109
19/20: loss=1.37334
20/20: loss=1.58556

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 698.13it/s]
Recorded results/billsum/pilots/billsum-lora-1564994f1226.json

Exit code: 0
Console log saved: /kaggle/working/week2_evidence/lora_20261112_061639.txt


In [22]:
import json
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

report_path = (
    project
    / "results/billsum/pilots/billsum-lora-1564994f1226.json"
)
report = json.loads(report_path.read_text())

for field in [
    "passed",
    "status",
    "successful_optimizer_steps",
    "steady_memory",
    "mean_steady_step_seconds",
    "checkpoint_bytes",
    "reload_max_abs_diff",
    "reload_tolerance"
]:
    print(field, ":", report.get(field, "not recorded"))

zip_path = Path("/kaggle/working/week2_evidence_after_lora.zip")

with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for folder in [
        project / "results/billsum",
        Path("/kaggle/working/week2_evidence")
    ]:
        for file in sorted(folder.rglob("*")):
            if file.is_file():
                archive.write(
                    file,
                    file.relative_to("/kaggle/working")
                )

    archive.write(
        project / "src/lora_budget/models/qwen.py",
        "source_snapshot/qwen.py"
    )

print("\nBackup created:", zip_path)
print("Includes current reports, logs, and the QLoRA code fix.")
print("Excludes model checkpoints and raw datasets.")
display(FileLink(str(zip_path)))

passed : True
status : completed
successful_optimizer_steps : 20
steady_memory : {'allocated_bytes': 3482987008, 'reserved_bytes': 4150263808}
mean_steady_step_seconds : 14.099776101733308
checkpoint_bytes : 29071512
reload_max_abs_diff : 0.0
reload_tolerance : 0.001

Backup created: /kaggle/working/week2_evidence_after_lora.zip
Includes current reports, logs, and the QLoRA code fix.
Excludes model checkpoints and raw datasets.


/kaggle/working/week2_evidence_after_lora.zip

In [23]:
import subprocess
import sys
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log_path = setup_logs.parent / f"dora_{stamp}.txt"

command = [
    sys.executable, "-u",
    "scripts/pilot_billsum.py",
    "--config", "configs/billsum/dora.yaml",
    "--steps", "20",
    "--seed", "42"
]

with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=str(project),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    exit_code = process.wait()

print("\nExit code:", exit_code)
print("Console log saved:", log_path)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 679.35it/s]
1/20: loss=1.54409
2/20: loss=1.56071
3/20: loss=1.39341
4/20: loss=1.68953
5/20: loss=1.44567
6/20: loss=1.62952
7/20: loss=1.64961
8/20: loss=1.63928
9/20: loss=1.69609
10/20: loss=1.24867
11/20: loss=1.44149
12/20: loss=1.80795
13/20: loss=1.58082
14/20: loss=1.70051
15/20: loss=1.56269
16/20: loss=1.27769
17/20: loss=1.63526
18/20: loss=1.45576
19/20: loss=1.36828
20/20: loss=1.58002

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 720.42it/s]
Recorded results/billsum/pilots/billsum-dora-5ccc88dbe43a.json

Exit code: 0
Console log saved: /kaggle/working/week2_evidence/dora_20261112_062908.txt


In [25]:
import json
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

report_path = (
    project
    / "results/billsum/pilots/billsum-dora-5ccc88dbe43a.json"
)
report = json.loads(report_path.read_text())

for field in [
    "passed",
    "status",
    "successful_optimizer_steps",
    "steady_memory",
    "mean_steady_step_seconds",
    "checkpoint_bytes",
    "reload_max_abs_diff",
    "reload_tolerance"
]:
    print(field, ":", report.get(field, "not recorded"))

zip_path = Path("/kaggle/working/week2_evidence_after_dora.zip")

with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for folder in [
        project / "results/billsum",
        Path("/kaggle/working/week2_evidence")
    ]:
        for file in sorted(folder.rglob("*")):
            if file.is_file():
                archive.write(
                    file,
                    file.relative_to("/kaggle/working")
                )

    archive.write(
        project / "src/lora_budget/models/qwen.py",
        "source_snapshot/qwen.py"
    )

print("\nBackup created:", zip_path)
print("Includes current reports, logs, and the QLoRA code fix.")
print("Excludes model checkpoints and raw datasets.")
display(FileLink(str(zip_path)))

passed : True
status : completed
successful_optimizer_steps : 20
steady_memory : {'allocated_bytes': 3487670784, 'reserved_bytes': 4206886912}
mean_steady_step_seconds : 24.32942265173333
checkpoint_bytes : 30310927
reload_max_abs_diff : 0.0
reload_tolerance : 0.001

Backup created: /kaggle/working/week2_evidence_after_dora.zip
Includes current reports, logs, and the QLoRA code fix.
Excludes model checkpoints and raw datasets.


/kaggle/working/week2_evidence_after_dora.zip

In [26]:
import subprocess
import sys
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log_path = setup_logs.parent / f"qlora_{stamp}.txt"

command = [
    sys.executable, "-u",
    "scripts/pilot_billsum.py",
    "--config", "configs/billsum/qlora.yaml",
    "--steps", "20",
    "--seed", "42"
]

with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=str(project),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    exit_code = process.wait()

print("\nExit code:", exit_code)
print("Console log saved:", log_path)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 349.49it/s]
1/20: loss=1.63331
2/20: loss=1.64276
3/20: loss=1.48325
4/20: loss=1.76126
5/20: loss=1.51677
6/20: loss=1.72074
7/20: loss=1.73374
8/20: loss=1.74607
9/20: loss=1.78593
10/20: loss=1.32840
11/20: loss=1.51294
12/20: loss=1.90046
13/20: loss=1.65936
14/20: loss=1.76572
15/20: loss=1.63009
16/20: loss=1.33302
17/20: loss=1.70196
18/20: loss=1.52081
19/20: loss=1.43203
20/20: loss=1.65959

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 395.87it/s]
Recorded results/billsum/pilots/billsum-qlora-b6bcc21e3505.json

Exit code: 0
Console log saved: /kaggle/working/week2_evidence/qlora_20261112_064135.txt


In [27]:
import json
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

report_path = (
    project
    / "results/billsum/pilots/billsum-qlora-b6bcc21e3505.json"
)
report = json.loads(report_path.read_text())

for field in [
    "passed",
    "status",
    "successful_optimizer_steps",
    "steady_memory",
    "mean_steady_step_seconds",
    "checkpoint_bytes",
    "reload_max_abs_diff",
    "reload_tolerance"
]:
    print(field, ":", report.get(field, "not recorded"))

# Record the installed software and the local code change.
for filename, command in [
    ("pip_freeze.txt", [sys.executable, "-m", "pip", "freeze"]),
    ("git_commit.txt", ["git", "rev-parse", "HEAD"]),
    ("git_diff.patch", ["git", "diff"]),
    ("git_status.txt", ["git", "status", "--short"])
]:
    result = subprocess.run(
        command,
        cwd=str(project),
        capture_output=True,
        text=True,
        check=True
    )
    (setup_logs / filename).write_text(result.stdout)

zip_path = Path("/kaggle/working/week2_evidence_all_four.zip")

with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for folder in [
        project / "results/billsum",
        Path("/kaggle/working/week2_evidence")
    ]:
        for file in sorted(folder.rglob("*")):
            if file.is_file():
                archive.write(
                    file,
                    file.relative_to("/kaggle/working")
                )

    archive.write(
        project / "src/lora_budget/models/qwen.py",
        "source_snapshot/qwen.py"
    )

print("\nBackup created:", zip_path)
print("Includes reports, logs, environment details, and the code fix.")
print("Excludes model checkpoints and raw datasets.")
display(FileLink(str(zip_path)))

passed : True
status : completed
successful_optimizer_steps : 20
steady_memory : {'allocated_bytes': 3544192000, 'reserved_bytes': 4206886912}
mean_steady_step_seconds : 16.15466087879995
checkpoint_bytes : 29071512
reload_max_abs_diff : 0.0
reload_tolerance : 0.001

Backup created: /kaggle/working/week2_evidence_all_four.zip
Includes reports, logs, environment details, and the code fix.
Excludes model checkpoints and raw datasets.


/kaggle/working/week2_evidence_all_four.zip